# 🏺 Tumulus LiDAR Detector — interactive demo

Detects **burial mounds (tumuli)** in 0.5 m airborne LiDAR — **no local install**.

**How to use:**
1. **Runtime → Run all** (runs the default area first), **or**
2. run cell **1** once, then in cell **2** type your own centre coordinates + box size and run it.

⚠️ **Coverage is Romania only** (ANCPI LAKI III). Use coordinates with **longitude ~20–29, latitude ~43–48**. Points outside Romania have no LiDAR tiles and will report no map.

You get: a **heatmap** of the scanned area, close-ups of the kept candidates, a table with the **exact coordinates** of every detected mound, and a **downloadable CSV** of coordinates + detection metrics. ~1–3 min (faster on a GPU runtime: *Runtime → Change runtime type → T4 GPU*).

Repo: https://github.com/ObuObuHub/tumulus-lidar-detector · *Form is not confirmation — only fieldwork settles a tumulus.*

In [ ]:
# 1) Get the code + model, install the one extra dependency (torch/numpy/Pillow are on Colab).
!git clone -q https://github.com/ObuObuHub/tumulus-lidar-detector.git
%cd tumulus-lidar-detector
!pip install -q pyproj

In [ ]:
# 2) ✏️ Pick a centre point (decimal degrees) and the scan-box size, then run this cell.
#    Coverage is ROMANIA only (LAKI III): longitude ~20-29, latitude ~43-48.
longitude = 23.522  #@param {type:"number"}
latitude  = 44.043  #@param {type:"number"}
area_km   = 2       #@param {type:"slider", min:1, max:8, step:1}

#  Downloads the public LiDAR tiles, renders hillshade, runs the CNN, applies the shape filters.
!python tools/scan_zone.py {longitude} {latitude} {area_km}

### Heatmap & detected candidates
**Green circles = kept candidates** (probable mounds); grey × = suppressed false positives. The board below shows a close-up of each kept candidate (only when there is at least one).

In [ ]:
import os
from IPython.display import Image, display
if os.path.exists('review/zone_view.jpg'):
    display(Image('review/zone_view.jpg'))       # heatmap of the scanned area
    if os.path.exists('review/zone_board.jpg'):
        display(Image('review/zone_board.jpg'))   # close-up crops (only when candidates were kept)
    else:
        print('No candidates kept in this area - no close-up board. The heatmap above is the LiDAR terrain.')
else:
    print('No map produced: the scan found no LiDAR tiles. Coverage is ROMANIA only (LAKI III) - use lon ~20-29, lat ~43-48. See the message in the scan cell above.')

### Detected mounds — coordinates, metrics & downloadable CSV

In [ ]:
import os, pandas as pd
from IPython.display import HTML, display

if not os.path.exists('/tmp/zone_dets.csv'):
    print('No detections file - the scan produced no output (coordinates outside Romania/LAKI III coverage, or a tile-download hiccup). Try lon ~20-29, lat ~43-48.')
else:
    det = pd.read_csv('/tmp/zone_dets.csv')
    # coordinates + full technical detection data for each kept mound
    mounds = det[det['keep'] == 1][['lon', 'lat', 'score', 'coh', 'pgate']].sort_values('score', ascending=False).reset_index(drop=True)
    mounds.insert(0, 'id', range(1, len(mounds) + 1))
    print(f'{len(mounds)} probable mound(s) detected - coordinates + detection metrics (highest score first):')
    shown = mounds.copy()
    shown['map'] = shown.apply(lambda r: f'<a href="https://www.google.com/maps?q={r.lat},{r.lon}" target="_blank">view</a>', axis=1)
    display(HTML(shown.to_html(escape=False, index=False)))
    # write a CSV with coordinates + technical detection data, then offer it for download
    mounds.to_csv('detected_mounds.csv', index=False)
    print('CSV columns: id, lon, lat, score (CNN confidence 0-1), coh (coherence - lower = more dome-like), pgate (curvature-gate prob - higher = smoother dome)')
    try:
        from google.colab import files
        files.download('detected_mounds.csv')   # triggers a browser download
    except Exception:
        print('detected_mounds.csv saved - see the Files panel on the left.')

Scan a different place: edit the three values in step 2 and re-run cells 2 → end (keep it inside Romania).

To check accuracy against your own confirmed mounds, run `tools/benchmark.py your_truth.csv combined_cnn.pt` (CSV columns `lon,lat`). Romanian site coordinates are withheld here for looting safety — supply your own, or a public set.